# Corpus build

In [50]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import os
import json

import yaml

In [2]:
base_path = f"{os.getcwd()}/Job/MELI"
raw_docs_path = "docs_raw"

# Taken from EDA
def list_files(path):

    lst = os.listdir(path)
    files = []    

    for f in lst:
        if os.path.isfile(f"{path}/{f}"):
            dtype = f"{path}/{f}"
            files.append(dtype)        

        elif os.path.isdir(f"{path}/{f}"):            

            files = files + list_files(f"{path}/{f}")

    return files

## For Markdown

In [3]:
# Important: possible problem escaping the double/single quotation marks.

import re

def escape_markdown(text: str) -> str:
    # Standard Markdown special characters
    escape_chars = r"_*[]()~`>#+-=|{}.!\\'"
    # Prepend a backslash to every special character found in the string
    return re.sub(f'([{re.escape(escape_chars)}])', r'\\\1', text)


In [31]:
# Replace special tags

def replace_code_chunks(content:str, code_chunk:str = "```"):

    starting_ind = 0

    while code_chunk in content:

        # Open
        if starting_ind == 0:
            content = content.replace(code_chunk, "<code_tkn>", 1)
            starting_ind = 1
        else:
            content = content.replace(code_chunk, "</code_tkn>", 1)
            starting_ind = 0

    return content   


# Method for storing the data

def save_entries(data, exit_path):

    with open(exit_path, "a", encoding="utf-8") as file:

        for e in data:
            file.write(json.dumps(e) + "\n")

    return

def is_indentation_line(line, patterns, check_digits = True):

    # Skip lines
    idt_flag = False    

    for idt in patterns:
        if line.startswith(idt):
            idt_flag = True
            break    
        
    if check_digits:

        if line.startswith(('0', '1', '2', '3', '4', '5', '6', '7', '8', '9')):
            idt_flag = True

    return idt_flag

def is_code_line(line, tokens):

    is_code = False    

    for idt in list(tokens.values()):
        if line.startswith(idt):
            is_code = True
            break      
            
    return is_code

def get_code_lines(lines, tokens):
    # Array

    total_lines = len(lines)

    open_token = False
    closed = False
    code_lines = [False] * total_lines

    for i in range(total_lines):
        if tokens["start"] in lines[i]:
            code_lines[i] = True
            open_token = True

        if tokens["end"] in lines[i]:
            code_lines[i] = True
            open_token = False


        # 
        if open_token == True:
            code_lines[i] = True  


    return code_lines

def join_lines(lines_arr, sep = " "):

    lines_str = []

    for l in lines_arr:
        lines_str.append(escape_markdown(l))

    joint_line = sep.join(lines_str)

    return joint_line

In [5]:
def generate_markdown_corpus(file_path:str, project_name:str, version:str, exit_path: str):

    # Excluded tokens
    code_tokens = {"start": "<code_tkn>", 
                   "end": "</code_tkn>"}

    # Read file
    with open(file_path, "r", encoding="utf-8") as file:
        content = file.read()

    # Replace code chunks <--- Here we can add any other method we need (also in a sequential array).
    content = replace_code_chunks(content)


    # Split by title
    title_orders = ["# ", "## ", "### ", "#### ", "##### "] 

    # Special indentations
    special_indts = ["-", "*"]

    in_code_block = False

    prev_category = 0
    parent_section = {}

    corpus = []

    section_name = ""

    content_lines = content.splitlines()
    total_lines = len(content_lines)

    # Check if they are indentation lines
    is_indent = []
    for line in content_lines:
        line = line.strip()
        is_indent.append(is_indentation_line(line, special_indts, True))

    # Check if is a code block
    is_code = get_code_lines(content_lines, code_tokens)

    print(len(content_lines) == len(is_code))
    

    # Create the corpus
    for li, line in enumerate(content_lines):
        line = line.strip()      

        # if line == "":
        #     continue
        

        if "<code_tkn>" in line:
            in_code_block = True

        elif "</code_tkn>" in line:
            in_code_block = False

        # Skip when is a indent line
        if is_indent[li] == True:
            continue

        if is_code[li] == True:
            continue        
        
        next_batch_lines = [line]

        # Indent lines        
        for nl_ind in range(li + 1, total_lines):
            if is_indent[nl_ind] == True:
                next_batch_lines.append(content_lines[nl_ind])                
            else:
                break

        # Code block lines
        for nl_ind in range(li + 1, total_lines):

            if is_code[nl_ind]:
                next_batch_lines.append(content_lines[nl_ind])              
                   
            else:
                break

          

        # Build sections
        is_section = False
        for i, t in enumerate(title_orders):

            if line[0: len(t)] == t and in_code_block == False:

                is_section = True

                if i >= prev_category:
                    parent_section[i] = line.replace(t, "")

                    section_name = [escape_markdown(s) for s in list(parent_section.values())]        


        if is_section == False:              

            entry = {
                "Name": f"{project_name}",
                "Path": f"{file_path}",
                "Version": f"{version}",
                "section": section_name,
                "content": join_lines(next_batch_lines).strip()
            }

            if entry["content"] != "":
                corpus.append(entry)
                # print(entry)


    save_entries(corpus, exit_path)

    print(f"Successfully generated {len(corpus)} entries.")
    # print(corpus)

                
    return            


### Simple methods for extraction

**Note:** Be careful with this methods, here they work because **ALL** folders share the same structure.

In [23]:
def get_project_name(file_path, base_folder_name, splitter_chr = "/"):

    subdirs = file_path.split(splitter_chr)

    name = ""

    for i, s in enumerate(subdirs):
        if s == base_folder_name:
            name = subdirs[i+1]
            break

    return name

def get_project_version(file_path, project_folder_name, splitter_chr = "/"):

    subdirs = file_path.split(splitter_chr)

    name = ""

    for i, s in enumerate(subdirs):
        if s == project_folder_name:
            name = subdirs[i+1]
            break
            


    return name


In [7]:
# Test
project_names_dir = "docs_raw"

file_path =  "/home/manuel/Job/MELI/docs_raw/2p-revenue-optimizer-api/0.2.2/guide/README.md"
p_name = get_project_name(file_path, project_names_dir)
p_version = get_project_version(file_path, p_name)

exit_path = "/home/manuel/Job/MELI/Solution/corpus/corpus.jsonl"

generate_markdown_corpus(file_path, p_name, p_version, exit_path)

True
Successfully generated 21 entries.


In [ ]:
docs_raw_path = f"{base_path}/{raw_docs_path}"

project_names_dir = "docs_raw"

exit_path_md = "/home/manuel/Job/MELI/Solution/corpus/corpus_md.jsonl"


for f in list_files(docs_raw_path):
    if f[-3:] == ".md":

        file_path =  f
        p_name = get_project_name(file_path, project_names_dir)
        p_version = get_project_version(file_path, p_name)

        

        generate_markdown_corpus(file_path, p_name, p_version, exit_path_md)

True
Successfully generated 21 entries.
True
Successfully generated 12 entries.
True
Successfully generated 13 entries.
True
Successfully generated 21 entries.
True
Successfully generated 9 entries.
True
Successfully generated 12 entries.
True
Successfully generated 13 entries.
True
Successfully generated 40 entries.
True
Successfully generated 40 entries.
True
Successfully generated 0 entries.
True
Successfully generated 15 entries.
True
Successfully generated 1 entries.
True
Successfully generated 2 entries.
True
Successfully generated 4 entries.
True
Successfully generated 9 entries.
True
Successfully generated 1 entries.
True
Successfully generated 5 entries.
True
Successfully generated 4 entries.
True
Successfully generated 5 entries.
True
Successfully generated 9 entries.
True
Successfully generated 17 entries.
True
Successfully generated 15 entries.
True
Successfully generated 12 entries.
True
Successfully generated 56 entries.
True
Successfully generated 62 entries.
True
Succes

## For JSON

In [61]:
def generate_json_corpus(file_path, p_name, p_version, exit_path):

    # Read file
    with open(file_path, "r", encoding="utf-8") as file:
        content = file.read()
        content_lines = content.splitlines()

    entries = []

    for line in content_lines:

        json_content = line[line.index('{'):line.index('}')+1]

        try:
            json.loads(json_content)

            entry = {
                "Name": f"{p_name}",
                "Path": f"{file_path}",
                "Version": f"{p_version}",
                "section": "JSON",
                "content": escape_markdown(json_content)
            }

            entries.append(entry)

        except Exception as e:
            entry = None    

    save_entries(entries, exit_path)
 
    return entries

In [62]:
# Keep it simple

exit_path_json = '/home/manuel/Job/MELI/Solution/corpus/corpus_json.jsonl'

docs_raw_path = f"{base_path}/{raw_docs_path}"

project_names_dir = "docs_raw"

for f in list_files(docs_raw_path):
    if f[-5:] == ".json":

        file_path =  f
        p_name = get_project_name(file_path, project_names_dir)
        p_version = get_project_version(file_path, p_name)


        generate_json_corpus(file_path, p_name, p_version, exit_path_json)

        


## For Yaml

In [66]:
def generate_yaml_corpus(file_path, p_name, p_version, exit_path):

    entries = []

    with open("/home/manuel/Job/MELI/docs_raw/zenith-keeper/1.1.6/specs/swagger.yaml", 'r') as file:
        config = yaml.safe_load(file)

    norm_json = pd.json_normalize(config).to_dict()


    for k in list(norm_json.keys()):

        if type(norm_json[k]) is dict:
            content = json.dumps(norm_json[k])
            


        entry = {
                "Name": f"{p_name}",
                "Path": f"{file_path}",
                "Version": f"{p_version}",
                "section": k.split('.'),
                "content": escape_markdown(content)
            }
        
        entries.append(entry)

    save_entries(entries, exit_path)       

    return 

In [67]:
# Keep it simple

exit_path_yaml = '/home/manuel/Job/MELI/Solution/corpus/corpus_yaml.jsonl'

docs_raw_path = f"{base_path}/{raw_docs_path}"

project_names_dir = "docs_raw"

for f in list_files(docs_raw_path):
    if f[-5:] == ".yaml":
        file_path =  f
        p_name = get_project_name(file_path, project_names_dir)
        p_version = get_project_version(file_path, p_name)


        generate_yaml_corpus(file_path, p_name, p_version, exit_path_yaml)

## Join all generated files and add an ID

In [69]:
corpus_files = [exit_path_md, exit_path_json, exit_path_yaml]

exit_path_corpus = "/home/manuel/Job/MELI/Solution/corpus/corpus.jsonl"

In [84]:
# Read file

content = []

for f in corpus_files:

    with open(f, "r", encoding="utf-8") as file:
        tmp_c = file.read()

        content = content + tmp_c.splitlines()

# Add Ids
updated_entries = []

for i in range(len(content)):

    entry = json.loads(content[i])

    id = f"{entry['Name']}_{i}"

    updated_entry = {'Id': id, **entry}
    updated_entries.append(updated_entry)

# Validate IDs

ids = [x['Id'] for x in updated_entries]

if len(np.unique(ids)) == len(content):
    print(f"Ids are ok to go")

    save_entries(updated_entries, exit_path_corpus)

Ids are ok to go
